In [1]:
import os
from typing import List

import numpy as np
import pandas as pd
import scanpy as sc

In [2]:
def gene_selection(
    gene_df: pd.DataFrame, 
    reference_df: pd.DataFrame
    ):

    reference = reference_df["gene_name"]
    to_fill_cols = list(set(reference) - set(gene_df.columns))

    padding_df = pd.DataFrame(
        np.zeros((gene_df.shape[0], len(to_fill_cols))),
        columns=to_fill_cols,
        index=gene_df.index
    )

    X_df = pd.DataFrame(
        np.concatenate([df.values for df in [gene_df, padding_df]], axis=1),
        index=gene_df.index,
        columns=gene_df.columns.tolist() + padding_df.columns.tolist()
    )

    X_df = X_df[reference]

    # var = pd.DataFrame(index=X_df.columns)
    # set_to_fill_cols = set(to_fill_cols)

    # var["mask"] = [1 if i in set_to_fill_cols else 0 for i in var.index]

    return X_df, to_fill_cols


def normalize(X: np.ndarray):
    X = X / X.sum(1, keepdims=True) * 10_000
    X = np.log1p(X)

    return X


In [3]:
reference_path = os.path.join("..", "data", "gene-reference-table.tsv")

reference_df = pd.read_csv(reference_path, sep="\t")
reference_df.head()

,gene_name,index
0,A1BG,0
1,A1CF,1
2,A2M,2
3,A2ML1,3
4,A3GALT2,4


In [4]:
len(reference_df)

19264

In [5]:
expression_path = os.path.join("..", "data", "raw", "gene-expression.csv")

os.path.isfile(expression_path)

True

In [6]:
expression_df = pd.read_csv(expression_path, sep=",", index_col=0)

expression_df.head()

,LASP1,HOXA11,CREBBP,ETV1,GAS7,CD79B,PAX7,BTK,BRCA1,WAS,...,NCKIPSD,MTCP1,DDX3X,FANCG,SSX2,ETV5,CEBPA,LSM14A,CUX1,C15orf65
ACH-000828,9.393476,0.042644,3.935460,0.871844,0.070389,0.084064,0.000000,0.056584,3.339137,0.150560,...,4.071248,3.119356,6.849374,4.355439,0.000000,0.137504,1.769772,6.501598,4.700994,2.295723
ACH-000568,7.638074,0.056584,3.427606,0.201634,1.794936,0.739848,0.042644,0.333424,3.193772,0.815575,...,4.084064,4.634593,5.671576,5.525443,0.056584,2.195348,0.124328,5.811214,3.590961,1.550901
ACH-000560,5.728193,6.001352,5.032542,5.018812,0.432959,0.250962,0.000000,0.263034,4.678635,1.292782,...,5.057450,3.468583,6.617798,6.425761,0.000000,5.203201,1.922198,7.581351,5.320124,1.438293
ACH-000561,6.037163,1.565597,4.262283,0.790772,1.257011,0.028569,0.056584,0.042644,3.442280,0.286881,...,3.400538,3.407353,6.154211,4.794936,0.000000,3.984589,1.028569,6.533719,5.132166,2.144046
ACH-000562,7.050502,0.014355,3.360364,0.879706,0.084064,0.137504,0.000000,0.042644,4.939227,0.286881,...,4.125982,4.047015,6.281884,5.853497,0.056584,3.757023,0.056584,5.912171,4.877744,0.815575


In [7]:
X_df, to_fill_cols = gene_selection(expression_df, reference_df)

X_df.head()

,A1BG,A1CF,A2M,A2ML1,A3GALT2,A4GALT,A4GNT,AAAS,AACS,AADAC,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
ACH-000828,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ACH-000568,0.0,0.028569,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ACH-000560,0.0,0.042644,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ACH-000561,0.0,0.042644,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ACH-000562,0.0,0.042644,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [8]:
X = X_df.values

X = normalize(X)

normalized_df = pd.DataFrame(X, index=X_df.index, columns=X_df.columns)
normalized_df.head()

,A1BG,A1CF,A2M,A2ML1,A3GALT2,A4GALT,A4GNT,AAAS,AACS,AADAC,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
ACH-000828,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ACH-000568,0.0,0.122203,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ACH-000560,0.0,0.152391,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ACH-000561,0.0,0.160657,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ACH-000562,0.0,0.161598,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
save_dir = os.path.join("..", "data", "raw")

normalized_df.to_csv(os.path.join(save_dir, "gene-expression-normalized.csv"), index=True)